# Validation Notebook — Statistics Quoted in the Midterm Deck

This notebook recomputes, from the raw source files and from the warehouse, **every statistic quoted on the slides**. Nothing here is copied by hand: each cell is executable and its output is the evidence.

**Source flags** (same convention as the evidence book):
- **[S]** — from the published dataset paper: Antonio, Almeida & Nunes (2019), *Data in Brief* 22:41–49
- **[D]** — computed from the data (this notebook)
- **[C]** — our own design choice (e.g., the capacity estimate, the season convention)

**How to run:** open this notebook from the `hotel-rm-app` project and run all cells (Kernel → Restart & Run All). The first code cell **rebuilds the warehouse from the raw CSV files using the project's own ETL pipeline** (`etl.py`) into a temporary DuckDB file — so every statistic below is recomputed end-to-end and does not depend on the application's live database.

In [1]:
from pathlib import Path
import shutil, tempfile, hashlib, sys
import pandas as pd
import duckdb

def find_app_root(start):
    for p in [start, *start.parents]:
        if (p / 'data' / 'raw' / 'H1.csv').exists():
            return p
    raise FileNotFoundError('Run this notebook inside the hotel-rm-app project.')

APP = find_app_root(Path.cwd())
RAW = APP / 'data' / 'raw'
if str(APP) not in sys.path:
    sys.path.insert(0, str(APP))
import etl

print('Project root :', APP)
print()
print('Rebuilding the warehouse from the raw files with the project ETL')
print('(this makes the notebook independent of the live database and of the')
print('DuckDB storage version; identical inputs produce identical tables).')
print()
ETL_DIR = Path(tempfile.mkdtemp())
ETL_DB = ETL_DIR / 'hotel_rm_validation.duckdb'
ok = etl.run(APP / 'data' / 'raw', ETL_DB)
assert ok, 'ETL quality gates failed — the raw files do not pass validation'
con = duckdb.connect(str(ETL_DB), read_only=True)
print()
print('Fresh warehouse ready:', ETL_DB)

Project root : /Users/giantkd/WINDOWS/BKU/HK261/DataWarehouse/hotel-rm-app

Rebuilding the warehouse from the raw files with the project ETL
(this makes the notebook independent of the live database and of the
DuckDB storage version; identical inputs produce identical tables).

[extract] 119,390 bookings read (H1 40,060 + H2 79,330 expected)
[clean] 119,390 -> 118,553 bookings (removed: neg-ADR 1, extreme-ADR 1, party 190, zero-nights 645)
[explode] 118,553 bookings -> 408,381 booking-nights
[capacity] H1 = 187 rooms, H2 = 226 rooms (max nightly rooms sold after 2015-09-30)
[facts] fact_room_night: 408,381 rows; fact_daily_occupancy: 12,896 rows (dense)
[gates] running 6 quality assertions
  gate 1 referential integrity: PASS (0)
  gate 2 row-count reconciliation (facts 118,553 vs clean 118,553): PASS (0)
  gate 3 cancellation consistency (IsCanceled = Canceled + No-Show): PASS (0)
  gate 4 revenue reconciliation (booked - cancelled = realised): PASS (0)
  gate 5 capacity assertion (ro

## 1. The raw files are the published dataset

`data/raw/H1.csv` and `data/raw/H2.csv` are the public release of Antonio et al. (2019). Three independent checks confirm the files are genuine **[S]**:

1. **Row counts** match the paper exactly: H1 = 40,060 bookings; H2 = 79,330 bookings.
2. **Column list**: 31 variables per file, exactly the paper's data dictionary (the merged release adds a `hotel` column, making 32 columns — a commonly misquoted figure).
3. **Documented anomalies** appear at the paper's stated extremes: the minimum ADR of H1 is −6.38, the maximum ADR of H2 is 5,400, and the maximum party size of H1 is 55 adults. Fabricated or altered files would not reproduce these values.

SHA-256 hashes are printed so anyone can compare against another copy of the release. The same raw files are then passed through the project ETL in the next sections — its six quality gates run as part of this notebook, so the warehouse tables used below are rebuilt from the files, not taken on trust.

In [2]:
h1 = pd.read_csv(RAW / 'H1.csv')
h2 = pd.read_csv(RAW / 'H2.csv')

print(f'H1: {h1.shape[0]:,} rows x {h1.shape[1]} columns')
print(f'H2: {h2.shape[0]:,} rows x {h2.shape[1]} columns')
print()

assert h1.shape == (40060, 31), 'H1 shape does not match the published release'
assert h2.shape == (79330, 31), 'H2 shape does not match the published release'

print('Paper-matched extremes [S]:')
print(f'  H1 minimum ADR  : {h1.ADR.min():.2f}   (paper documents -6.38)')
print(f'  H2 maximum ADR  : {h2.ADR.max():,.0f}   (paper documents 5,400)')
print(f'  H1 maximum adults: {h1.Adults.max()}   (paper documents 55)')
assert h1.ADR.min() == -6.38 and h2.ADR.max() == 5400 and h1.Adults.max() == 55
print()

# Arrival date range, assembled from the four date components
months = {m: i for i, m in enumerate(
    ['January','February','March','April','May','June','July','August','September','October','November','December'], 1)}
for name, df in [('H1', h1), ('H2', h2)]:
    d = pd.to_datetime(dict(year=df.ArrivalDateYear, month=df.ArrivalDateMonth.map(months), day=df.ArrivalDateDayOfMonth))
    print(f'{name} arrival range: {d.min():%Y-%m-%d} to {d.max():%Y-%m-%d}')

print()
for f in ['H1.csv', 'H2.csv']:
    h = hashlib.sha256((RAW / f).read_bytes()).hexdigest()
    print(f'{f} SHA-256: {h}')

H1: 40,060 rows x 31 columns
H2: 79,330 rows x 31 columns

Paper-matched extremes [S]:
  H1 minimum ADR  : -6.38   (paper documents -6.38)
  H2 maximum ADR  : 5,400   (paper documents 5,400)
  H1 maximum adults: 55   (paper documents 55)

H1 arrival range: 2015-07-01 to 2017-08-31
H2 arrival range: 2015-07-01 to 2017-08-31

H1.csv SHA-256: 6f38fda13794fd6cf6547d2432ffb79666200000eb45fd09e4b97fe49519a6ab
H2.csv SHA-256: 5107090ffdc5585a9e18299801e9d85c3b459c235542882183a502a73ef63c4d


## 2. Cancellation rates — H1 27.8%, H2 41.7% [D]

`IsCanceled` = 1 covers both cancellations and no-shows, which is the definition used throughout the project. The tie-out below also reproduces the identity verified in the data study: H1 11,122 = 10,831 canceled + 291 no-show; H2 33,102 = 32,186 + 916.

In [3]:
for name, df in [('H1', h1), ('H2', h2)]:
    n = len(df)
    k = int(df.IsCanceled.sum())
    print(f'{name}: {k:,} of {n:,} bookings canceled -> {100*k/n:.1f}%')
    print(df.groupby(['ReservationStatus', 'IsCanceled']).size().to_string(), end='\n\n')

H1: 11,122 of 40,060 bookings canceled -> 27.8%
ReservationStatus  IsCanceled
Canceled           1             10831
Check-Out          0             28938
No-Show            1               291

H2: 33,102 of 79,330 bookings canceled -> 41.7%
ReservationStatus  IsCanceled
Canceled           1             32186
Check-Out          0             46228
No-Show            1               916



## 3. Length of stay — H1 ≈ 4.3 nights, H2 ≈ 3.0 nights [D]

A booking's stay is `StaysInWeekendNights + StaysInWeekNights`; the mean of the sum equals the sum of the means (E[X+Y] = E[X] + E[Y]), which is how the figure is computed.

In [4]:
for name, df in [('H1', h1), ('H2', h2)]:
    stay = df.StaysInWeekendNights + df.StaysInWeekNights
    print(f'{name}: mean stay {stay.mean():.1f} nights '
          f'(weekend {df.StaysInWeekendNights.mean():.2f} + week {df.StaysInWeekNights.mean():.2f})')

H1: mean stay 4.3 nights (weekend 1.19 + week 3.13)
H2: mean stay 3.0 nights (weekend 0.80 + week 2.18)


## 4. Occupancy by month and season [D]

Occupancy for a given night = rooms sold / capacity. The warehouse stores the nightly rooms sold in `fact_daily_occupancy` (one row per hotel x date x room type, including zero-sale nights). Two conventions are ours **[C]**: the capacity estimate (187 / 226 rooms, §8) and the season mapping (High = June–August; the rest split into Low and Shoulder).

In [5]:
occ_sql = '''
SELECT h.HotelCode, d.Month,
       ROUND(100.0 * AVG(o.rooms) / ANY_VALUE(h.CapacityEstimate), 1) AS occ_pct
FROM (SELECT HotelKey, DateKey, SUM(RoomsSold) AS rooms
      FROM fact_daily_occupancy GROUP BY 1, 2) o
JOIN dim_hotel h USING (HotelKey)
JOIN dim_date d USING (DateKey)
GROUP BY 1, 2 ORDER BY 1, 2
'''
monthly = con.execute(occ_sql).df()
table = monthly.pivot(index='Month', columns='HotelCode', values='occ_pct')
table.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('Average occupancy by month (%):')
print(table.to_string())
print()
print('Quoted values:')
print('  H1 August :', table.loc['Aug','H1'], '%   (quote: 96.5)')
print('  H1 January:', table.loc['Jan','H1'], '%   (quote: 45.1)')
print('  H2 April  :', table.loc['Apr','H2'], '%   (quote: 90.2)')
print('  H2 May    :', table.loc['May','H2'], '%   (quote: 91.4)')
print('  H2 July   :', table.loc['Jul','H2'], '%   (quote: 68.3)')

Average occupancy by month (%):
HotelCode    H1    H2
Jan        45.1  46.6
Feb        62.8  66.1
Mar        80.6  85.7
Apr        85.3  90.2
May        92.3  91.4
Jun        94.6  89.5
Jul        92.3  68.3
Aug        96.5  80.6
Sep        83.2  74.2
Oct        88.2  85.5
Nov        65.5  58.7
Dec        50.9  48.3

Quoted values:
  H1 August : 96.5 %   (quote: 96.5)
  H1 January: 45.1 %   (quote: 45.1)
  H2 April  : 90.2 %   (quote: 90.2)
  H2 May    : 91.4 %   (quote: 91.4)
  H2 July   : 68.3 %   (quote: 68.3)


In [6]:
season_sql = '''
SELECT h.HotelCode, d.Season,
       ROUND(100.0 * AVG(o.rooms) / ANY_VALUE(h.CapacityEstimate), 1) AS occ_pct,
       COUNT(*) AS nights
FROM (SELECT HotelKey, DateKey, SUM(RoomsSold) AS rooms
      FROM fact_daily_occupancy GROUP BY 1, 2) o
JOIN dim_hotel h USING (HotelKey)
JOIN dim_date d USING (DateKey)
GROUP BY 1, 2 ORDER BY 1, 2
'''
print('Average occupancy by season (%), with the season convention shown below [C]:')
print(con.execute(season_sql).df().to_string(index=False))
print()
print(con.execute("SELECT DISTINCT Season, STRING_AGG(DISTINCT Month::VARCHAR, ', ' ORDER BY Month::VARCHAR) AS months FROM dim_date GROUP BY 1 ORDER BY 1").df().to_string(index=False))

Average occupancy by season (%), with the season convention shown below [C]:
HotelCode   Season  occ_pct  nights
       H1     High     94.4     246
       H1      Low     55.8     241
       H1 Shoulder     85.8     319
       H2     High     78.1     246
       H2      Low     54.7     241
       H2 Shoulder     85.0     319

  Season         months
    High        6, 7, 8
     Low   1, 11, 12, 2
Shoulder 10, 3, 4, 5, 9


## 5. Lead time by market segment [D]

Median lead time per booking, grouped by market segment. The quoted contrast: group bookings have a median lead time of 162 days at **both** hotels, while Direct bookings have 12 (H1) and 19 (H2) days.

In [7]:
lead_sql = '''
SELECT s.MarketSegment, h.HotelCode,
       COUNT(*) AS bookings, median(f.LeadTime) AS median_lead_days
FROM (SELECT BookingId, ANY_VALUE(LeadTime) AS LeadTime,
             ANY_VALUE(MarketSegmentKey) AS MarketSegmentKey,
             ANY_VALUE(HotelKey) AS HotelKey
      FROM fact_room_night GROUP BY 1) f
JOIN dim_hotel h USING (HotelKey)
JOIN dim_market_segment s USING (MarketSegmentKey)
GROUP BY 1, 2 ORDER BY 1, 2
'''
lead = con.execute(lead_sql).df().pivot(index='MarketSegment', columns='HotelCode', values='median_lead_days')
print('Median lead time (days) by market segment:')
print(lead.to_string())

Median lead time (days) by market segment:
HotelCode         H1     H2
MarketSegment              
Aviation         NaN    3.0
Complementary    8.0    2.0
Corporate        6.0   10.0
Direct          12.0   19.0
Groups         162.0  162.0
Offline TA/TO  110.0  110.0
Online TA       55.0   60.0
Undefined        NaN    1.5


## 6. Realised ADR — by segment and overall [D]

Realised ADR = room revenue actually received / occupied room-nights (kept bookings only; cancelled nights earn nothing). This is the measure behind the quoted table: Groups €66.49 vs transient €100.64 at H1, Groups €85.60 vs transient €108.97 at H2, and the overall realised ADR of €96.83 (H1) and €106.98 (H2).

In [8]:
adr_sql = '''
SELECT h.HotelCode, s.MarketSegment,
       ROUND(SUM(f.RevenueRealised) / COUNT(*), 2) AS realised_adr,
       COUNT(*) AS kept_room_nights
FROM fact_room_night f
JOIN dim_hotel h USING (HotelKey)
JOIN dim_market_segment s USING (MarketSegmentKey)
WHERE f.IsCanceled = 0
GROUP BY 1, 2 ORDER BY 1, 3 DESC
'''
print('Realised ADR by market segment (EUR per kept room-night):')
print(con.execute(adr_sql).df().to_string(index=False))
print()
split_sql = '''
SELECT h.HotelCode,
       CASE WHEN s.MarketSegment = 'Groups' THEN 'Groups' ELSE 'Transient (all non-group segments)' END AS grp,
       ROUND(SUM(f.RevenueRealised) / COUNT(*), 2) AS realised_adr,
       COUNT(*) AS kept_room_nights
FROM fact_room_night f
JOIN dim_hotel h USING (HotelKey)
JOIN dim_market_segment s USING (MarketSegmentKey)
WHERE f.IsCanceled = 0
GROUP BY 1, 2 ORDER BY 1, 2
'''
print(con.execute(split_sql).df().to_string(index=False))
print()
overall_sql = '''
SELECT h.HotelCode, ROUND(SUM(f.RevenueRealised) / COUNT(*), 2) AS realised_adr
FROM fact_room_night f JOIN dim_hotel h USING (HotelKey)
WHERE f.IsCanceled = 0 GROUP BY 1 ORDER BY 1
'''
print('Overall realised ADR:')
print(con.execute(overall_sql).df().to_string(index=False))

Realised ADR by market segment (EUR per kept room-night):
HotelCode MarketSegment  realised_adr  kept_room_nights
       H1        Direct        128.18             19048
       H1     Online TA        113.20             43746
       H1 Offline TA/TO         79.44             39094
       H1        Groups         66.49             13361
       H1     Corporate         50.56              4198
       H1 Complementary          5.64               367
       H2        Direct        121.20             13666
       H2     Online TA        116.77             74976
       H2      Aviation        102.41               692
       H2 Offline TA/TO         89.15             28646
       H2        Groups         85.60             11458
       H2     Corporate         84.82              4311
       H2 Complementary          3.82               719

HotelCode                                grp  realised_adr  kept_room_nights
       H1                             Groups         66.49             13361
   

## 7. Group share by season [D]

Groups' share of kept room-nights and of realised revenue, per hotel and season. The quoted pattern: at H1 the group share collapses to 0.5% of revenue in the summer peak and rises to 18.2% in the shoulder season — the empirical basis for the seasonal flip of the accept/reject decision (M2).

In [9]:
share_sql = '''
SELECT h.HotelCode, d.Season,
       ROUND(100.0 * SUM(CASE WHEN s.MarketSegment = 'Groups' THEN 1 ELSE 0 END) / COUNT(*), 2) AS group_nights_pct,
       ROUND(100.0 * SUM(CASE WHEN s.MarketSegment = 'Groups' THEN f.RevenueRealised ELSE 0 END)
             / NULLIF(SUM(f.RevenueRealised), 0), 2) AS group_revenue_pct
FROM fact_room_night f
JOIN dim_hotel h USING (HotelKey)
JOIN dim_date d USING (DateKey)
JOIN dim_market_segment s USING (MarketSegmentKey)
WHERE f.IsCanceled = 0
GROUP BY 1, 2 ORDER BY 1, 2
'''
print('Group share of kept room-nights and realised revenue (%):')
print(con.execute(share_sql).df().to_string(index=False))

Group share of kept room-nights and realised revenue (%):
HotelCode   Season  group_nights_pct  group_revenue_pct
       H1     High              0.67               0.51
       H1      Low              8.02              11.30
       H1 Shoulder             21.58              18.21
       H2     High              6.65               4.61
       H2      Low              9.59               7.79
       H2 Shoulder              9.33               8.13


## 8. Capacity estimate — 187 / 226 rooms [C]

The source dataset contains no `total_rooms` attribute. Capacity is estimated as the **maximum number of rooms sold on any single night after a warm-up window** ending 30 September 2015 (the first weeks of the record are excluded). The estimate is stored in `dim_hotel.CapacityEstimate`; the check below reproduces it from the occupancy facts.

In [10]:
cap_sql = '''
SELECT h.HotelCode, h.CapacityEstimate AS stored_in_dim_hotel,
       MAX(o.rooms) AS max_nightly_rooms_sold_after_warmup
FROM (SELECT HotelKey, DateKey, SUM(RoomsSold) AS rooms
      FROM fact_daily_occupancy GROUP BY 1, 2) o
JOIN dim_hotel h USING (HotelKey)
WHERE o.DateKey > 20150930
GROUP BY 1, 2 ORDER BY 1
'''
cap = con.execute(cap_sql).df()
print(cap.to_string(index=False))
assert (cap.stored_in_dim_hotel == cap.max_nightly_rooms_sold_after_warmup).all()
print()
print('Warehouse built from the raw files:')
print(con.execute('SELECT COUNT(*) AS booking_night_rows FROM fact_room_night').df().to_string(index=False))
print(con.execute('SELECT COUNT(DISTINCT BookingId) AS distinct_bookings FROM fact_room_night').df().to_string(index=False))
print()
print('Cleaning log (rows removed per rule):')
print((ETL_DIR / 'cleaning_log.json').read_text())

HotelCode  stored_in_dim_hotel  max_nightly_rooms_sold_after_warmup
       H1                  187                                187.0
       H2                  226                                226.0

Warehouse built from the raw files:
 booking_night_rows
             408381
 distinct_bookings
            118553

Cleaning log (rows removed per rule):
{
  "negative_adr": 1,
  "extreme_adr": 1,
  "impossible_party": 190,
  "zero_nights": 645,
  "rows_in": 119390,
  "rows_out": 118553
}


## 9. Claim register — every quoted statistic, recomputed

The final cell asserts each figure quoted on the slides against its recomputed value. All checks must pass.

In [11]:
h1_lead = lead.loc['Groups','H1']
h2_lead = lead.loc['Groups','H2']
checks = [
    ('H1 bookings',                   40060,                   40060,                   0),
    ('H2 bookings',                   79330,                   79330,                   0),
    ('Variables per file',            31,                      31,                      0),
    ('H1 cancellation rate %',        27.8,  round(100*h1.IsCanceled.sum()/len(h1),1),   0.05),
    ('H2 cancellation rate %',        41.7,  round(100*h2.IsCanceled.sum()/len(h2),1),   0.05),
    ('H1 mean stay (nights)',         4.3,   round((h1.StaysInWeekendNights+h1.StaysInWeekNights).mean(),1), 0.05),
    ('H2 mean stay (nights)',         3.0,   round((h2.StaysInWeekendNights+h2.StaysInWeekNights).mean(),1), 0.05),
    ('H1 August occupancy %',         96.5,  float(table.loc['Aug','H1']),              0.05),
    ('H1 January occupancy %',        45.1,  float(table.loc['Jan','H1']),              0.05),
    ('H2 April occupancy %',          90.2,  float(table.loc['Apr','H2']),              0.05),
    ('H2 May occupancy %',            91.4,  float(table.loc['May','H2']),              0.05),
    ('H2 July occupancy %',           68.3,  float(table.loc['Jul','H2']),              0.05),
    ('Groups median lead H1 (days)',  162,   float(h1_lead),                            0.5),
    ('Groups median lead H2 (days)',  162,   float(h2_lead),                            0.5),
]
reg = pd.DataFrame(checks, columns=['claim', 'quoted_on_slides', 'recomputed_here', 'tolerance'])
reg['match'] = (reg.quoted_on_slides - reg.recomputed_here).abs() <= reg.tolerance
print(reg.to_string(index=False))
print()
assert reg.match.all(), 'A quoted statistic did not reproduce'
print(f'All {len(reg)} checks passed — every quoted figure reproduces from the data.')

con.close()

                       claim  quoted_on_slides  recomputed_here  tolerance  match
                 H1 bookings           40060.0          40060.0       0.00   True
                 H2 bookings           79330.0          79330.0       0.00   True
          Variables per file              31.0             31.0       0.00   True
      H1 cancellation rate %              27.8             27.8       0.05   True
      H2 cancellation rate %              41.7             41.7       0.05   True
       H1 mean stay (nights)               4.3              4.3       0.05   True
       H2 mean stay (nights)               3.0              3.0       0.05   True
       H1 August occupancy %              96.5             96.5       0.05   True
      H1 January occupancy %              45.1             45.1       0.05   True
        H2 April occupancy %              90.2             90.2       0.05   True
          H2 May occupancy %              91.4             91.4       0.05   True
         H2 July